In [ ]:
# SCENIC MAF-regulon analysis producing thesis figures 4.5.1 to 4.5.3.
"""
SCENIC MAF analysis — 3 final figures (PNG, dpi 300).
  Fig 1: MAF binarized regulon activity by timepoint_condition (square cells)
  Fig 2: MAF target connectivity network (Day 4, motif-validated)
  Fig 3: LYVE1 importance under MAF across Day 1-4
"""

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import loompy as lp
import ast
import networkx as nx
from sklearn import mixture
from pathlib import Path

# ===================================================================
# Paths
# ===================================================================
SCENIC_LOOM = '/home/yanjun2/lyve1project/scanpy/8_figures/Module5_scenic/inte_scenic.loom'
ADATA_PATH  = '/home/yanjun2/lyve1project/scanpy/8_figures/inte_palantir.h5ad'
REG_DIR     = '/home/yanjun2/lyve1project/scanpy/8_figures/Module5_scenic'
OUT_DIR     = Path('/home/yanjun2/lyve1project/scanpy/8_figures/Module5_scenic/MAF_figures')
OUT_DIR.mkdir(parents=True, exist_ok=True)

DPI = 300

# ===================================================================
# Load adata, build timepoint_condition
# ===================================================================
adata = sc.read(ADATA_PATH)
adata.obs['condition'] = (adata.obs['condition'].astype(str)
                          .astype('category'))
adata.obs['timepoint_condition'] = (
    adata.obs['timepoint'].astype(str) + '_' + adata.obs['condition'].astype(str)
).astype('category')
days = sorted(adata.obs['timepoint'].unique())
conds = ['MDI', 'MDIX']
order = [f'{d}_{c}' for d in days for c in conds
         if f'{d}_{c}' in adata.obs['timepoint_condition'].cat.categories]
adata.obs['timepoint_condition'] = adata.obs['timepoint_condition'].cat.reorder_categories(order)

# ===================================================================
# Load AUC and binarize (SCENIC author's GMM method + random_state)
# ===================================================================
lf = lp.connect(SCENIC_LOOM, mode='r+', validate=False)
auc_mtx = pd.DataFrame(lf.ca.RegulonsAUC, index=lf.ca.CellID)
auc_mtx.columns = auc_mtx.columns.str.replace(r'\(', '_(', regex=True)
assert (auc_mtx.index == adata.obs_names).all(), "Cell index mismatch!"

def _derive_threshold(auc_mtx, regulon_name, random_state=42):
    data = auc_mtx[regulon_name].values.reshape(-1, 1)
    gmm = mixture.GaussianMixture(n_components=2, covariance_type='full',
                                   random_state=random_state).fit(data)
    avgs = gmm.means_
    stds = np.sqrt(gmm.covariances_.reshape(-1, 1))
    idx = np.argmax(avgs)
    threshold = max(avgs[idx] - 2 * stds[idx], 0)
    idx = np.argmin(avgs)
    lower_bound = avgs[idx] + 2 * stds[idx]
    return float(max(lower_bound, threshold))

thresholds = pd.Series({c: _derive_threshold(auc_mtx, c) for c in auc_mtx.columns})
binarized = (auc_mtx > thresholds).astype(int)

# ===================================================================
# Figure 1: MAF binarized activity by timepoint_condition (square cells)
# ===================================================================
maf_col = [c for c in binarized.columns if c.startswith('MAF_(')][0]
df = pd.DataFrame({
    'binarized': binarized[maf_col].values,
    'group': adata.obs['timepoint_condition'].values,
})
maf_fraction = df.groupby('group', observed=True)['binarized'].mean().reindex(order)

# Square cells: figsize width matches n_cols, height matches n_rows (1 row)
n_cols = len(maf_fraction)
cell_size = 0.7  # inches per cell
fig, ax = plt.subplots(figsize=(n_cols * cell_size + 1.8, cell_size + 0.8))
sns.heatmap(
    maf_fraction.to_frame().T,
    cmap='RdBu_r', center=maf_fraction.mean(),
    annot=True, fmt='.2f',
    cbar_kws={'label': 'Fraction MAF+', 'shrink': 0.8},
    ax=ax, linewidths=0.5, linecolor='white',
    square=True,
)
ax.set_yticklabels(['MAF regulon active'], rotation=0)
ax.set_xlabel('')
ax.set_title('MAF regulon activity (binarized)', fontsize=11, pad=8)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig1a_MAF_binarized_heatmap.png', dpi=DPI, bbox_inches='tight')
plt.close()

# Barplot version
fig, ax = plt.subplots(figsize=(5.5, 3.2))
colors = ['#888888' if g.split('_')[1] == 'MDI' else '#E65100' for g in maf_fraction.index]
ax.bar(maf_fraction.index, maf_fraction.values,
       color=colors, edgecolor='black', linewidth=0.8)
ax.set_ylabel('Fraction MAF+ cells')
ax.set_ylim(0, 1)
ax.set_title('MAF regulon activity by timepoint × condition')
plt.xticks(rotation=45, ha='right')
sns.despine()
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig1b_MAF_binarized_bar.png', dpi=DPI, bbox_inches='tight')
plt.close()

# ===================================================================
# Figure 2: MAF connectivity network (Day 4, motif-validated)
# ===================================================================
def parse_regulon(reg_path, tf='MAF'):
    reg = pd.read_csv(reg_path, skiprows=3, header=None,
                      names=['TF', 'MotifID', 'AUC', 'NES', 'MotifSimQval',
                             'OrthologousIdentity', 'Annotation', 'Context',
                             'TargetGenes', 'RankAtMax'])
    targets = []
    for _, row in reg.query('TF == @tf').iterrows():
        try:
            targets.extend(ast.literal_eval(row['TargetGenes']))
        except (ValueError, SyntaxError):
            continue
    return (pd.DataFrame(targets, columns=['target', 'importance'])
            .groupby('target', as_index=False)['importance'].max()
            .sort_values('importance', ascending=False)
            .reset_index(drop=True))

maf_targets_day4 = parse_regulon(f'{REG_DIR}/reg_day4.csv', tf='MAF')

def plot_maf_network(targets_df, top_n=25, output='MAF_network.png',
                     highlight=None, tf='MAF'):
    tf_top = targets_df.nlargest(top_n, 'importance').copy()
    imp = tf_top['importance']
    tf_top['width'] = 1 + 4 * (imp - imp.min()) / (imp.max() - imp.min())
    G = nx.DiGraph()
    G.add_node(tf)
    for _, r in tf_top.iterrows():
        G.add_edge(tf, r['target'], weight=r['width'])
    pos = nx.shell_layout(G, [[tf], [n for n in G.nodes if n != tf]])
    highlight = set(highlight or [])
    node_colors = ['#2E7D32' if n == tf
                   else '#E65100' if n in highlight
                   else '#CE93D8' for n in G.nodes]
    node_sizes = [2500 if n == tf else 600 for n in G.nodes]
    widths = [d['weight'] for _, _, d in G.edges(data=True)]
    fig, ax = plt.subplots(figsize=(11, 9))
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes,
                           edgecolors='black', linewidths=1.2, ax=ax)
    nx.draw_networkx_edges(
        G, pos, width=widths, edge_color='#555555', alpha=0.7,
        arrows=True, arrowsize=18, arrowstyle='-|>',
        connectionstyle='arc3,rad=0.05', node_size=node_sizes,
        min_source_margin=20, min_target_margin=20, ax=ax,
    )
    nx.draw_networkx_labels(
        G, pos, font_size=11,
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1), ax=ax,
    )
    ax.set_title(f'{tf} regulon: top {top_n} motif-validated targets (Day 4)',
                 fontsize=14, pad=20)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(output, dpi=DPI, bbox_inches='tight')
    plt.close()

plot_maf_network(
    maf_targets_day4, top_n=25,
    output=OUT_DIR / 'fig2_MAF_network_day4.png',
    highlight=['LYVE1', 'F13A1', 'STAB1', 'CPT1B', 'MRC1', 'FOLR2'],
)

# ===================================================================
# Figure 3: LYVE1 importance trajectory (Day 1-4) — no CD163, no MAFB
# ===================================================================
def get_target_importance(reg_path, tf, target):
    df = parse_regulon(reg_path, tf=tf)
    row = df.query('target == @target')
    return float(row['importance'].values[0]) if not row.empty else 0.0

markers_to_track = ['LYVE1', 'F13A1', 'MRC1', 'FOLR2', 'STAB1']

trajectory = pd.DataFrame(
    {gene: [get_target_importance(f'{REG_DIR}/reg_day{d}.csv', 'MAF', gene)
            for d in range(1, 5)]
     for gene in markers_to_track},
    index=[f'Day{d}' for d in range(1, 5)],
)

fig, ax = plt.subplots(figsize=(6, 4.5))
for gene in markers_to_track:
    is_lyve1 = gene == 'LYVE1'
    ax.plot(trajectory.index, trajectory[gene],
            marker='o', markersize=8 if is_lyve1 else 5,
            linewidth=2.5 if is_lyve1 else 1.2,
            color='#E65100' if is_lyve1 else None,
            alpha=1.0 if is_lyve1 else 0.6,
            label=gene, zorder=3 if is_lyve1 else 2)
ax.set_xlabel('Differentiation timepoint')
ax.set_ylabel('Target importance in MAF regulon')
ax.set_title('Recruitment of LYVE1+ resident markers into MAF regulon')
ax.legend(loc='upper left', frameon=False, fontsize=9)
ax.grid(axis='y', alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig3_MAF_target_trajectory.png', dpi=DPI, bbox_inches='tight')
plt.close()

print('Done. Outputs in:', OUT_DIR)